In [1]:
import time
import pandas as pd
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer

c:\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MODELS = {
    "TinyLlama":
        "TinyLlama/TinyLlama-1.1B-Chat-v1.0",

    "Qwen2.5":
        "Qwen/Qwen2.5-1.5B-Instruct",

    "SmolLM2":
        "HuggingFaceTB/SmolLM2-1.7B-Instruct"
}

In [4]:
prompts = [

    "Explain overfitting in machine learning.",

    "What is the difference between precision and recall?",

    "Summarize the role of transformers in NLP.",

    "Why is statistical significance important in AI research?",

    "Explain the concept of uncertainty estimation."
]

In [7]:
def evaluate_model(model,model_path,prompt):

    start=time.time()

    tokenizer=AutoTokenizer.from_pretrained(model_path)

    model=AutoModelForCausalLM.from_pretrained(model_path,torch_dtype="auto")
    
    input=tokenizer(prompt,return_tensors="pt")

    output=model.generate(**input,max_new_tokens=100)
    
    end=time.time()
    
    response=tokenizer.decode(output[0],skip_special_tokens=True)    
    
    return response,end-start

In [8]:
results = []

for model_name, model_path in MODELS.items():

    for prompt in prompts:

        response, latency = evaluate_model(
            model_name,
            model_path,
            prompt
        )

        results.append({
            "model": model_name,
            "prompt": prompt,
            "latency": latency,
            "response_length": len(response),
            "response": response
        })

In [9]:
df=pd.DataFrame(results)

df.to_csv(
    "benchmark_results.csv",
    index=False
)

In [10]:
df.groupby("model")[
    ["latency","response_length"]
].mean()

,latency,response_length
model,,
Qwen2.5,124.730178,612.6
SmolLM2,106.936999,47.4
TinyLlama,1.345295,47.4
